In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [5]:
# 1. SORT DATA CORRECTLY
# Sort by Channel and Date to compare "Current" vs "Previous"
df_research_ready = pd.read_csv('../0_data/research_ready_videos.csv')
df_ml = df_research_ready.sort_values(['channel_id', 'upload_date']).copy()

# 2. CREATE "CATEGORY SWITCH" (The independent variable)
# Shift the category column down by 1 to get the "Previous Category"
df_ml['prev_category'] = df_ml.groupby('channel_id')['category'].shift(1)

# Create a Boolean Flag: 1 if changed, 0 if same (or first video)
df_ml['category_switch'] = np.where(
    (df_ml['prev_category'].notna()) & (df_ml['category'] != df_ml['prev_category']),
    1,
    0
)

# 3. SELECT FEATURES & TARGET
# Target: Log Views
y = df_ml['log_views']

# Features: Category Switch + The Category Itself (Control) + Channel Stats
# We use One-Hot Encoding for the 'category' column (e.g., is_Gaming, is_Music)
cat_dummies = pd.get_dummies(df_ml['category'], prefix='cat')

# Combine everything into X
X = pd.concat([df_ml[['category_switch', 'view_count']], cat_dummies], axis=1)

# Drop raw 'view_count' from X (we don't want to predict views using views)
# We keep it only if you want to use "Lagged Views" (Previous views) as a control,
# which is smart. Let's create Lagged Views:
X['prev_views'] = df_ml.groupby('channel_id')['log_views'].shift(1)
X = X.fillna(0) # Fill first video NaNs with 0
del X['view_count'] # Remove the dependent variable leak

# 4. SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape: ", X_test.shape)

Training Data Shape: (3314695, 17)
Testing Data Shape:  (828674, 17)


## Linear regression

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Train
lr = LinearRegression()
lr.fit(X_train, y_train)

# 2. Predict
y_pred_lr = lr.predict(X_test)

# 3. Evaluate
print("--- Linear Regression Results ---")
print(f"R-Squared: {r2_score(y_test, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")

# 4. Interpret
switch_coef = lr.coef_[0] # Assuming 'category_switch' is the first column
print(f"\nCoefficient for Category Switch: {switch_coef:.4f}")
if switch_coef < 0:
    print(f"INTERPRETATION: Switching categories implies a {abs(switch_coef)*100:.1f}% DROP in views.")
else:
    print(f"INTERPRETATION: Switching categories implies a {switch_coef*100:.1f}% RISE in views.")

--- Linear Regression Results ---
R-Squared: 0.5579
RMSE: 1.6490

Coefficient for Category Switch: -0.0552
INTERPRETATION: Switching categories implies a 5.5% DROP in views.


## Random forest

In [7]:
from sklearn.ensemble import RandomForestRegressor

# n_jobs=-1 uses all CPU cores (faster)
rf = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\n--- Random Forest Results ---")
print(f"R-Squared: {r2_score(y_test, y_pred_rf):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")


--- Random Forest Results ---
R-Squared: 0.5820
RMSE: 1.6033


## SVM

In [10]:
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# SVM requires scaling (normalization) to work well
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. dual=False:  Optimizes for "n_samples > n_features"
# 2. max_iter=5000: Gives the model more time to converge
svr = LinearSVR(
    dual=False, 
    loss='squared_epsilon_insensitive',
    max_iter=5000, 
    random_state=42
)

print("Training SVM... (This might take a minute)")
svr.fit(X_train_scaled, y_train)

y_pred_svr = svr.predict(X_test_scaled)

print("\n--- SVM (Linear) Results ---")
print(f"R-Squared: {r2_score(y_test, y_pred_svr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_svr)):.4f}")

Training SVM... (This might take a minute)

--- SVM (Linear) Results ---
R-Squared: 0.5579
RMSE: 1.6490
